In [1]:
from aviary.core import Environment, Message, Tool, ToolRequestMessage,ToolResponseMessage
from ldp.agent import Agent
from ldp.graph import LLMCallOp
import os
from helper.problem import examples
from helper.units import UNIT_ALIASES
from helper.tools import *
from typing import Any, Tuple, List
from pydantic import BaseModel, Field
from ldp.graph import LLMCallOp, OpResult, compute_graph

In [ ]:
os.environ["OPENAI_API_KEY"] = ""

In [ ]:
# --- Step 1: Science (more like chemistry) Environment -> ---
class ScienceEnv(Environment[ProblemState]):
    def __init__(self, problem: ProblemState):
          super().__init__()
          self.problem = problem
          self._oracle = None
        
    async def reset(self):
        self.tools = [Tool.from_function(fun) 
                      for _, fun in TOOLS.items()]
        print(f"[self.problem]: {self.problem}")
        prompt = self.problem.prompt or build_prompt(self.problem)
        print(f"[RESET] prompt: {prompt}")
        return [Message(content=prompt)], self.tools

    async def step(self, action):
        # 1) If it’s a tool request, run it and return *native* ToolResponseMessage(s)
        if isinstance(action, ToolRequestMessage) and action.tool_calls:
            raw = await self.exec_tool_calls(action)
            # raw may be one ToolResponseMessage or a list of them
            if isinstance(raw, ToolResponseMessage):
                obs = [raw]
            elif isinstance(raw, list) and all(isinstance(r, ToolResponseMessage) for r in raw):
                obs = raw
            else:
                # (shouldn’t happen—just in case)
                obs = [Message(content=str(raw))]
            return obs, 0.0, False, False

        # 2) Otherwise `action` is the final assistant Message—score and finish
        assert isinstance(action, Message) and action.content is not None
        reward = float(action.content.strip() == self.problem.expected.strip())
        return [action], reward, True, False


In [ ]:
# --- Step 2: Guided Agent ---
# can decide what tools to use at each step without using tool selector
# uses compute graph

class GuidedAgentState(BaseModel):
    messages: List[Message]
    tools:    List[Tool]

    def get_next_state(self, obs: List[Message]) -> "GuidedAgentState":
        new_msgs = list(self.messages)

        for item in obs:
            if isinstance(item, Message):
                new_msgs.append(item)

            elif hasattr(item, "content"):
                new_msgs.append(Message(content=item.content))

            elif isinstance(item, dict):
                new_msgs.append(Message(**item))

            else:
                continue

        return GuidedAgentState(messages=new_msgs, tools=self.tools)

class GuidedAgent(BaseModel, Agent[GuidedAgentState]):
    llm_model: dict[str, Any] = Field(
        default={
            "name": "gpt-4o-2024-08-06",
            "temperature": 0.1,
        }
    )
    guidelines_msg: Message = Field(
        default=Message(
            role="system",
            content=(
                "You are a scientific assistant.  You **must** use the provided tools for *every* calculation.\n"
                "Step by step:\n"
                "  1) To compute a mixture’s concentration, call **mix_concentration** with JSON args.\n"
                "  2) Observe the tool’s output (e.g. “0.175 M”).\n"
                "  3) To compute dilution volume, call **dilution** with JSON args using that exact output.\n"
                "  4) Observe and return that final tool output as your answer."
            )
        ),
        description="System‐level instructions for the LLM.",
    )

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self._llm_call_op = LLMCallOp()

    async def init_state(self, tools: List[Tool]) -> GuidedAgentState:
        return GuidedAgentState(messages=[self.guidelines_msg], tools=tools)

    @compute_graph()
    async def get_asv(self, agent_state: GuidedAgentState, obs: List[Message]) -> Tuple[
          OpResult[ToolRequestMessage], 
          GuidedAgentState,
          float                          
        ]:
        # 1) incorporate the new environment observation
        next_state = agent_state.get_next_state(obs)

        # 2) call the LLM to pick exactly one tool + args
        llm_result: OpResult[ToolRequestMessage] = await self._llm_call_op(
            self.llm_model,
            [self.guidelines_msg, *next_state.messages],
            next_state.tools
        )

        # 3) append the tool call into the agent’s message history
        next_state.messages.append(llm_result.value)

        # 4) return the op result
        return llm_result, next_state, 0.0

In [5]:
examples.append(
    ProblemState(
        prompt=(
            "You have 0.05 L of a 0.1 M solution mixed with 0.15 L of a 0.2 M solution. "
            "First compute the mixture concentration. Then dilute that mixture to 0.05 M—"
            "what final volume (in L) do you need?"
        ),
        params={"C1":0.1, "V1":0.05, "C2":0.2, "V2":0.15, "final_conc":0.05},
        expected="0.700 L"
    )
)


In [ ]:
agent = GuidedAgent()

for problem in examples[-1:]:
    env = ScienceEnv(problem)
    obs, tools = await env.reset()
    state = await agent.init_state(tools)

    print("\n--- Chain‐of‐Thought ---")
    while True:
        # 1) Ask the agent for its next move
        op_result, state, _ = await agent.get_asv(state, obs)
        action = op_result.value
        print(op_result)
        print(op_result.value)

        # 2) Execute in the env
        obs, _, _, _ = await env.step(action)
        tool_output = obs[0].content.strip()
        print("[Tool output]", tool_output)

        # 3) If the answer is here, stop
        if tool_output == problem.expected.strip():
            print("\n[Final answer]", tool_output)
            print("Expected:      ", problem.expected)
            print("Reward:        ", float(tool_output == problem.expected.strip()))
            break


[self.problem]: prompt='You have 0.05 L of a 0.1 M solution mixed with 0.15 L of a 0.2 M solution. First compute the mixture concentration. Then dilute that mixture to 0.05 M—what final volume (in L) do you need?' task=None params={'C1': 0.1, 'V1': 0.05, 'C2': 0.2, 'V2': 0.15, 'final_conc': 0.05} expected='0.700 L'
[RESET] prompt: You have 0.05 L of a 0.1 M solution mixed with 0.15 L of a 0.2 M solution. First compute the mixture concentration. Then dilute that mixture to 0.05 M—what final volume (in L) do you need?

--- Chain‐of‐Thought ---
Tool request message '' for tool calls: mix_concentration(C1='0.1', V1='0.05', C2='0.2', V2='0.15') [id=call_nevbqHCYBs4pxKZy2mKndV8k]
Tool request message '' for tool calls: mix_concentration(C1='0.1', V1='0.05', C2='0.2', V2='0.15') [id=call_nevbqHCYBs4pxKZy2mKndV8k]
[Tool output] 0.175 M
Tool request message '' for tool calls: dilution(initial_conc='0.175', initial_vol='0.2', final_conc='0.05') [id=call_gPKUcAFQPnusJNFDZbjjpCzH]
Tool request mes